In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="19wVSIhL149bRP3qj7rqEgXlDTfySxat2", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


# Notebook 2: Batch Size Experiments

**Vizuara AI Pods | GPU Programming Course | Pod 4: Ring-AllReduce, Batch Size, and Profiling**

---

In this notebook, we will run real training experiments to understand the relationship between batch size, learning rate, and convergence. You will learn:

1. How batch size affects training loss curves and convergence speed
2. Why learning rate must be scaled when the batch size changes
3. The difference between linear scaling and square root scaling rules
4. Why warmup is essential for large-batch training
5. How to find the practical "sweet spot" batch size

**Prerequisites:** Notebook 1 (ring-allreduce concepts), basic PyTorch knowledge.

**Estimated time:** 40-50 minutes

**Runtime:** T4 GPU recommended: Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 🎧 Transition: Setup Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
#@title 🎧 Code Walkthrough: Pip Install
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_pip_install.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
!pip install -q torch torchvision numpy matplotlib tqdm

In [ ]:
#@title 🎧 Code Walkthrough: Imports Device
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_imports_device.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from collections import defaultdict

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
#@title 🎧 Transition: Part1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_part1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: Setting Up the Experiment

We will train a small CNN on CIFAR-10. This is small enough to run quickly on Colab, but large enough to see real batch size effects.

### The Model

A simple but effective CNN with ~2.4M parameters.

In [ ]:
#@title 🎧 Code Walkthrough: Model Definition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_model_definition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimpleCNN(nn.Module):
    """A simple CNN for CIFAR-10 classification."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 -> 64 channels
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 32x32 -> 16x16

            # Block 2: 64 -> 128 channels
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16x16 -> 8x8

            # Block 3: 128 -> 256 channels
            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8x8 -> 4x4
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# Count parameters
model = SimpleCNN().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")
print(f"Gradient size: {num_params * 4 / 1e6:.1f} MB (float32)")
del model

In [ ]:
#@title 🎧 Code Walkthrough: Cifar10 Data
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_cifar10_data.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Load CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")

In [ ]:
#@title 🎧 Transition: Part2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_part2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: Training Function with Metrics Tracking

We need a training function that carefully tracks:
- Loss curve (per step and per epoch)
- Accuracy
- Throughput (samples/second)
- Wall-clock time to reach a target accuracy

This will let us make fair comparisons across batch sizes.

In [ ]:
#@title 🎧 Code Walkthrough: Train Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_train_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def train_with_config(
    batch_size: int,
    learning_rate: float,
    num_epochs: int,
    warmup_steps: int = 0,
    label: str = "",
):
    """
    Train a model with the given configuration and track metrics.

    Returns a dict of training metrics.
    """
    # Fresh model for each run
    torch.manual_seed(42)
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=256, shuffle=False,
        num_workers=2, pin_memory=True
    )

    metrics = {
        'label': label or f'bs={batch_size}, lr={learning_rate:.1e}',
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'step_losses': [],
        'epoch_losses': [],
        'epoch_accuracies': [],
        'throughputs': [],      # samples/sec per epoch
        'total_samples': 0,
        'total_time': 0,
    }

    global_step = 0
    start_time = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_samples = 0
        epoch_start = time.time()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

            # Warmup: linearly increase LR
            if global_step < warmup_steps and warmup_steps > 0:
                warmup_lr = learning_rate * (global_step + 1) / warmup_steps
                for pg in optimizer.param_groups:
                    pg['lr'] = warmup_lr
            elif global_step == warmup_steps and warmup_steps > 0:
                for pg in optimizer.param_groups:
                    pg['lr'] = learning_rate

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * inputs.size(0)
            epoch_samples += inputs.size(0)
            metrics['step_losses'].append(loss.item())
            global_step += 1

        epoch_time = time.time() - epoch_start
        avg_loss = epoch_loss / epoch_samples
        throughput = epoch_samples / epoch_time
        metrics['epoch_losses'].append(avg_loss)
        metrics['throughputs'].append(throughput)
        metrics['total_samples'] += epoch_samples

        # Evaluate
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, targets in test_loader:
                inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                correct += predicted.eq(targets).sum().item()
                total += targets.size(0)
        accuracy = 100.0 * correct / total
        metrics['epoch_accuracies'].append(accuracy)

        print(f"  Epoch {epoch+1}/{num_epochs}: loss={avg_loss:.4f}, acc={accuracy:.1f}%, "
              f"throughput={throughput:.0f} samples/s")

    metrics['total_time'] = time.time() - start_time

    del model
    torch.cuda.empty_cache()

    return metrics

In [ ]:
#@title 🎧 Transition: Part3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_part3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: Experiment 1 -- Same LR, Different Batch Sizes

First, let's see what happens when we increase the batch size **without** adjusting the learning rate. This demonstrates the problem that scaling rules solve.

In [ ]:
#@title 🎧 Code Walkthrough: Exp1 Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_exp1_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Experiment 1: Fixed learning rate, varying batch size
FIXED_LR = 0.01
NUM_EPOCHS = 10

batch_sizes = [64, 256, 1024, 4096]
results_fixed_lr = []

for bs in batch_sizes:
    print(f"\n{'='*60}")
    print(f"Training with batch_size={bs}, lr={FIXED_LR}")
    print(f"{'='*60}")
    metrics = train_with_config(
        batch_size=bs,
        learning_rate=FIXED_LR,
        num_epochs=NUM_EPOCHS,
        label=f'bs={bs}'
    )
    results_fixed_lr.append(metrics)

print("\nDone! All experiments complete.")

In [ ]:
#@title 🎧 What to Look For: Exp1 Visualize
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_exp1_visualize.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize Experiment 1 results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#1E88E5', '#43A047', '#FB8C00', '#E53935']

# Plot 1: Loss per step (first 500 steps)
ax = axes[0, 0]
for r, c in zip(results_fixed_lr, colors):
    max_steps = min(500, len(r['step_losses']))
    ax.plot(range(max_steps), r['step_losses'][:max_steps], alpha=0.7,
            color=c, linewidth=1.5, label=r['label'])
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss (First 500 Steps)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Epoch loss
ax = axes[0, 1]
for r, c in zip(results_fixed_lr, colors):
    ax.plot(range(1, len(r['epoch_losses']) + 1), r['epoch_losses'], 'o-',
            color=c, linewidth=2, markersize=6, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Loss per Epoch (Fixed LR)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Test accuracy per epoch
ax = axes[1, 0]
for r, c in zip(results_fixed_lr, colors):
    ax.plot(range(1, len(r['epoch_accuracies']) + 1), r['epoch_accuracies'], 'o-',
            color=c, linewidth=2, markersize=6, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy per Epoch (Fixed LR)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 4: Throughput
ax = axes[1, 1]
avg_throughputs = [np.mean(r['throughputs']) for r in results_fixed_lr]
bar_positions = range(len(batch_sizes))
bars = ax.bar(bar_positions, avg_throughputs, color=colors, alpha=0.8, edgecolor='black')
ax.set_xticks(bar_positions)
ax.set_xticklabels([f'bs={bs}' for bs in batch_sizes], fontsize=10)
ax.set_ylabel('Throughput (samples/sec)', fontsize=12)
ax.set_title('Average Training Throughput', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, tp in zip(bars, avg_throughputs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Experiment 1: Fixed Learning Rate, Varying Batch Size\n'
             f'(LR = {FIXED_LR})', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Observation: Larger batch sizes converge SLOWER with the same LR.")
print("This is because the learning rate is effectively too small for the batch size.")
print("Each gradient step with a larger batch is more accurate but takes the same size step.")

In [ ]:
#@title 🎧 Transition: Part4 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_part4_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Experiment 2 -- Learning Rate Scaling Rules

Now let's apply the two scaling rules from the article:

1. **Linear scaling (Goyal et al., 2017):** `lr_new = lr_base * (bs_new / bs_base)`
2. **Square root scaling (Hoffer et al., 2017):** `lr_new = lr_base * sqrt(bs_new / bs_base)`

We will use a base configuration of `batch_size=64, lr=0.01` and scale to larger batch sizes.

In [ ]:
#@title 🎧 Code Walkthrough: Exp2 Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_exp2_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
BASE_BS = 64
BASE_LR = 0.01
NUM_EPOCHS = 10

# Configurations to test
configs = [
    # (batch_size, lr, warmup_steps, label)
    (64,   BASE_LR, 0, 'Base: bs=64, lr=0.01'),
    (256,  BASE_LR * (256 / BASE_BS), 100, 'Linear: bs=256, lr=0.04'),
    (256,  BASE_LR * np.sqrt(256 / BASE_BS), 100, 'Sqrt: bs=256, lr=0.02'),
    (1024, BASE_LR * (1024 / BASE_BS), 200, 'Linear: bs=1024, lr=0.16'),
    (1024, BASE_LR * np.sqrt(1024 / BASE_BS), 200, 'Sqrt: bs=1024, lr=0.04'),
]

results_scaling = []

for bs, lr, warmup, label in configs:
    print(f"\n{'='*60}")
    print(f"{label} (warmup={warmup} steps)")
    print(f"{'='*60}")
    metrics = train_with_config(
        batch_size=bs,
        learning_rate=lr,
        num_epochs=NUM_EPOCHS,
        warmup_steps=warmup,
        label=label
    )
    results_scaling.append(metrics)

print("\nAll scaling experiments complete!")

In [ ]:
#@title 🎧 What to Look For: Exp2 Visualize
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_exp2_visualize.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize scaling experiment results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#1E88E5', '#43A047', '#81C784', '#E53935', '#EF9A9A']

# Plot 1: Epoch loss
ax = axes[0]
for r, c in zip(results_scaling, colors):
    ax.plot(range(1, len(r['epoch_losses']) + 1), r['epoch_losses'], 'o-',
            color=c, linewidth=2, markersize=5, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Average Loss', fontsize=12)
ax.set_title('Loss with Scaling Rules', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy
ax = axes[1]
for r, c in zip(results_scaling, colors):
    ax.plot(range(1, len(r['epoch_accuracies']) + 1), r['epoch_accuracies'], 'o-',
            color=c, linewidth=2, markersize=5, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy with Scaling Rules', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

# Plot 3: Accuracy vs wall-clock time
ax = axes[2]
for r, c in zip(results_scaling, colors):
    total_time = r['total_time']
    time_per_epoch = total_time / len(r['epoch_accuracies'])
    times = [(i + 1) * time_per_epoch for i in range(len(r['epoch_accuracies']))]
    ax.plot(times, r['epoch_accuracies'], 'o-',
            color=c, linewidth=2, markersize=5, label=r['label'])
ax.set_xlabel('Wall-Clock Time (s)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle('Experiment 2: Learning Rate Scaling Rules', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\nSummary Table:")
print(f"{'Config':<40} {'Final Loss':>10} {'Final Acc':>10} {'Time':>8}")
print("-" * 70)
for r in results_scaling:
    print(f"{r['label']:<40} {r['epoch_losses'][-1]:>10.4f} {r['epoch_accuracies'][-1]:>9.1f}% {r['total_time']:>7.1f}s")

In [ ]:
#@title 🎧 Transition: Part5 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_part5_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: The Importance of Warmup

Let's directly compare training with and without warmup for a large batch size + high learning rate.

In [ ]:
#@title 🎧 Code Walkthrough: Exp3 Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_exp3_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Experiment 3: Warmup vs No Warmup
LARGE_BS = 1024
LARGE_LR = BASE_LR * (LARGE_BS / BASE_BS)  # Linear scaling -> 0.16
NUM_EPOCHS = 10

configs_warmup = [
    (LARGE_BS, LARGE_LR, 0,   f'No warmup (bs={LARGE_BS}, lr={LARGE_LR:.2f})'),
    (LARGE_BS, LARGE_LR, 50,  f'50-step warmup (bs={LARGE_BS}, lr={LARGE_LR:.2f})'),
    (LARGE_BS, LARGE_LR, 200, f'200-step warmup (bs={LARGE_BS}, lr={LARGE_LR:.2f})'),
    (LARGE_BS, LARGE_LR, 500, f'500-step warmup (bs={LARGE_BS}, lr={LARGE_LR:.2f})'),
]

results_warmup = []

for bs, lr, warmup, label in configs_warmup:
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    metrics = train_with_config(
        batch_size=bs,
        learning_rate=lr,
        num_epochs=NUM_EPOCHS,
        warmup_steps=warmup,
        label=label
    )
    results_warmup.append(metrics)

print("\nWarmup experiments complete!")

In [ ]:
#@title 🎧 What to Look For: Exp3 Visualize
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_exp3_visualize.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize warmup results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']

# Loss per step (first 300 steps to see warmup effect)
ax = axes[0]
for r, c in zip(results_warmup, colors):
    max_steps = min(300, len(r['step_losses']))
    ax.plot(range(max_steps), r['step_losses'][:max_steps],
            color=c, linewidth=1.5, alpha=0.8, label=r['label'])
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Early Training Loss (Warmup Effect)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)

# Final accuracy comparison
ax = axes[1]
for r, c in zip(results_warmup, colors):
    ax.plot(range(1, len(r['epoch_accuracies']) + 1), r['epoch_accuracies'], 'o-',
            color=c, linewidth=2, markersize=6, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy: Warmup Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f'Experiment 3: Warmup Effect (bs={LARGE_BS}, lr={LARGE_LR:.2f})',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey Insight: Without warmup, large LR causes instability early in training.")
print("A warmup period lets the model find a reasonable loss basin before taking large steps.")

In [ ]:
#@title 🎧 Transition: Part6 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_part6_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 6: Gradient Noise Analysis

Why does batch size affect training? The fundamental reason is **gradient noise**. With a small batch, the gradient estimate is noisy (high variance). With a large batch, it is more accurate (low variance) but each step processes more samples for the same update.

Let's measure gradient noise directly.

In [ ]:
#@title 🎧 Code Walkthrough: Gradient Noise Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_gradient_noise_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def measure_gradient_noise(model, dataset, batch_sizes, num_samples=20):
    """
    Measure gradient variance for different batch sizes.

    For each batch size, compute the gradient multiple times on different
    mini-batches and measure the variance across these estimates.
    """
    criterion = nn.CrossEntropyLoss()
    results = {}

    for bs in batch_sizes:
        loader = DataLoader(dataset, batch_size=bs, shuffle=True,
                           num_workers=0, drop_last=True)

        # Collect gradient estimates
        all_grads = []
        for i, (inputs, targets) in enumerate(loader):
            if i >= num_samples:
                break
            inputs, targets = inputs.to(device), targets.to(device)

            model.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()

            # Concatenate all parameter gradients into one vector
            grad_vec = torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None])
            all_grads.append(grad_vec.detach().cpu())

        grads = torch.stack(all_grads)
        grad_mean = grads.mean(dim=0)
        grad_var = grads.var(dim=0).mean().item()  # Average variance across parameters
        grad_norm = grad_mean.norm().item()
        signal_to_noise = grad_norm**2 / (grad_var + 1e-10)

        results[bs] = {
            'variance': grad_var,
            'mean_norm': grad_norm,
            'snr': signal_to_noise,
        }
        print(f"  bs={bs:>5}: grad_variance={grad_var:.6f}, |grad_mean|={grad_norm:.4f}, SNR={signal_to_noise:.1f}")

    return results


# Measure gradient noise at the start of training
torch.manual_seed(42)
noise_model = SimpleCNN().to(device)

print("Measuring gradient noise for different batch sizes:")
print("=" * 70)
batch_sizes_noise = [16, 32, 64, 128, 256, 512, 1024, 2048]
noise_results = measure_gradient_noise(noise_model, train_dataset, batch_sizes_noise)

del noise_model
torch.cuda.empty_cache()

In [ ]:
#@title 🎧 What to Look For: Gradient Noise Visualize
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_gradient_noise_visualize.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize gradient noise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bs_list = list(noise_results.keys())
variances = [noise_results[bs]['variance'] for bs in bs_list]
norms = [noise_results[bs]['mean_norm'] for bs in bs_list]
snrs = [noise_results[bs]['snr'] for bs in bs_list]

# Plot 1: Gradient variance vs batch size
ax = axes[0]
ax.loglog(bs_list, variances, 'o-', color='#E53935', linewidth=2, markersize=8)
# Theoretical line: variance ~ 1/batch_size
theoretical = [variances[0] * bs_list[0] / bs for bs in bs_list]
ax.loglog(bs_list, theoretical, '--', color='gray', linewidth=1.5, label='1/bs (theory)')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Gradient Variance', fontsize=12)
ax.set_title('Gradient Variance vs Batch Size', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Gradient mean norm
ax = axes[1]
ax.semilogx(bs_list, norms, 'o-', color='#1E88E5', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Gradient Norm', fontsize=12)
ax.set_title('Mean Gradient Norm vs Batch Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Signal-to-noise ratio
ax = axes[2]
ax.loglog(bs_list, snrs, 'o-', color='#43A047', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Signal-to-Noise Ratio', fontsize=12)
ax.set_title('Gradient SNR vs Batch Size', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.suptitle('Gradient Noise Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key Insight: Gradient variance decreases as ~1/batch_size (CLT).")
print("Larger batches give more accurate gradients, justifying larger learning rates.")
print("But beyond some point, the gradient is already accurate enough -- diminishing returns.")

In [ ]:
#@title 🎧 Transition: Part7 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_part7_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 7: Simulating Distributed Batch Sizes

In distributed training with N GPUs, the effective batch size is `N * local_batch_size`. Let's simulate what happens when we scale from 1 GPU to 8, 16, or 64 GPUs by using the equivalent global batch size on our single GPU.

In [ ]:
#@title 🎧 Code Walkthrough: Distributed Simulation Run
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_distributed_simulation_run.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Simulate scaling from 1 GPU to multiple GPUs
LOCAL_BS = 128  # Fixed local batch size per GPU
BASE_LR_SIM = 0.01
NUM_EPOCHS_SIM = 8

gpu_configs = [
    (1,  LOCAL_BS * 1,  'sqrt'),   # 1 GPU:  global_bs = 128
    (4,  LOCAL_BS * 4,  'sqrt'),   # 4 GPUs: global_bs = 512
    (8,  LOCAL_BS * 8,  'sqrt'),   # 8 GPUs: global_bs = 1024
    (16, LOCAL_BS * 16, 'sqrt'),   # 16 GPUs: global_bs = 2048
]

results_scaling_sim = []

for num_gpus, global_bs, rule in gpu_configs:
    k = global_bs / (LOCAL_BS * 1)  # scaling factor
    if rule == 'linear':
        lr = BASE_LR_SIM * k
    else:
        lr = BASE_LR_SIM * np.sqrt(k)

    warmup = max(50, int(100 * np.sqrt(k)))
    label = f'{num_gpus} GPU(s): global_bs={global_bs}, lr={lr:.4f}'

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")

    metrics = train_with_config(
        batch_size=global_bs,
        learning_rate=lr,
        num_epochs=NUM_EPOCHS_SIM,
        warmup_steps=warmup,
        label=label
    )
    metrics['num_gpus'] = num_gpus
    results_scaling_sim.append(metrics)

print("\nScaling simulation complete!")

In [ ]:
#@title 🎧 What to Look For: Distributed Simulation Visualize
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_distributed_simulation_visualize.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize scaling results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#1E88E5', '#43A047', '#FB8C00', '#E53935']

# Accuracy over epochs
ax = axes[0]
for r, c in zip(results_scaling_sim, colors):
    ax.plot(range(1, len(r['epoch_accuracies']) + 1), r['epoch_accuracies'], 'o-',
            color=c, linewidth=2, markersize=6, label=r['label'])
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Epoch (Simulated Multi-GPU)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Effective throughput (what it would be with real multi-GPU)
ax = axes[1]
gpu_nums = [r['num_gpus'] for r in results_scaling_sim]
# Single-GPU throughput for the base case
base_throughput = np.mean(results_scaling_sim[0]['throughputs'])
# Simulated multi-GPU throughput (ideal linear scaling)
ideal_throughputs = [base_throughput * n for n in gpu_nums]
# Realistic throughputs (accounting for ~10% communication overhead per doubling)
realistic_throughputs = [base_throughput * n * (0.92 ** np.log2(max(n, 1))) for n in gpu_nums]

ax.plot(gpu_nums, ideal_throughputs, 'o--', color='gray', linewidth=1.5,
        markersize=8, label='Ideal (linear) scaling')
ax.plot(gpu_nums, realistic_throughputs, 's-', color='#1E88E5', linewidth=2,
        markersize=8, label='Realistic (with comm. overhead)')
ax.set_xlabel('Number of GPUs', fontsize=12)
ax.set_ylabel('Throughput (samples/sec)', fontsize=12)
ax.set_title('Expected Multi-GPU Throughput', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insight: With proper LR scaling and warmup, we can scale to")
print("many GPUs while maintaining convergence quality.")
print("Communication overhead slightly reduces the theoretical speedup.")

## Exercises


In [ ]:
#@title 🎧 Before You Start: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_24_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 1: Find the Critical Batch Size

The **critical batch size** is the point beyond which doubling the batch size no longer halves the number of steps to reach a target loss. Train the model with batch sizes [64, 128, 256, 512, 1024, 2048, 4096] using sqrt scaling, and for each one record how many optimizer steps it takes to reach a loss of 0.5. Plot steps-to-target vs batch size. Where is the critical batch size?

In [ ]:
# TODO Exercise 1: Find the critical batch size
#
# For each batch size:
#   1. Use sqrt scaling: lr = 0.01 * sqrt(bs / 64)
#   2. Use warmup of 100 * sqrt(bs / 64) steps
#   3. Train until the running average loss drops below 0.5
#      (or for max 15 epochs, whichever comes first)
#   4. Record the number of optimizer steps taken
#
# Then plot:
#   - X-axis: batch size (log scale)
#   - Y-axis: steps to reach target loss (log scale)
#   - Also plot the "ideal" line: steps ~ 1/batch_size
#   - The critical batch size is where the actual curve starts
#     diverging from the ideal line

TARGET_LOSS = 0.5
batch_sizes_crit = [64, 128, 256, 512, 1024, 2048, 4096]

# TODO: Implement the experiment
# steps_to_target = []
# for bs in batch_sizes_crit:
#     ...

print("TODO: Implement the critical batch size experiment")
print("Hint: Track step_losses and find the first step where")
print("a running average (window=50) drops below the target.")

In [ ]:
#@title 🎧 Before You Start: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_25_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Compare Optimizers

Different optimizers respond differently to batch size scaling. Compare SGD with momentum, Adam, and AdamW at batch sizes [64, 512, 2048]. For Adam/AdamW, the common practice is to NOT scale the learning rate (or use very mild scaling). Test this and compare convergence.

In [ ]:
# TODO Exercise 2: Compare optimizer behavior across batch sizes
#
# For each (optimizer, batch_size) combination:
#   - SGD+momentum: use sqrt scaling (lr_base=0.01)
#   - Adam: use fixed lr=1e-3 (or very mild scaling)
#   - AdamW: use fixed lr=1e-3 with weight_decay=0.01
#
# Train for 10 epochs each and compare final accuracy.
# Which optimizer is most robust to batch size changes?

# Hint: Modify train_with_config to accept an optimizer_type parameter,
# or create separate training functions for each optimizer.

print("TODO: Compare SGD, Adam, and AdamW across batch sizes")
print("Key question: Is Adam's adaptive learning rate effectively")
print("doing its own per-parameter 'scaling' that makes it more")
print("robust to batch size changes?")

In [ ]:
#@title 🎧 Before You Start: Todo3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_26_todo3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 3: Gradient Accumulation

When your GPU memory is too small for a large batch, you can simulate a large batch by **accumulating gradients** over multiple micro-batches before taking an optimizer step. Implement gradient accumulation and verify that it produces the same training dynamics as a single large batch.

In [ ]:
# TODO Exercise 3: Implement gradient accumulation
#
# Goal: Show that training with:
#   - batch_size=256, accumulation_steps=1 (effective bs=256)
#   - batch_size=64,  accumulation_steps=4 (effective bs=256)
#   - batch_size=32,  accumulation_steps=8 (effective bs=256)
# all produce similar training curves.
#
# The gradient accumulation loop looks like:
#
# optimizer.zero_grad()
# for micro_step in range(accumulation_steps):
#     outputs = model(micro_batch)
#     loss = criterion(outputs, targets) / accumulation_steps
#     loss.backward()  # gradients accumulate
# optimizer.step()

print("TODO: Implement gradient accumulation training")
print("This is essential when your GPU cannot fit the desired batch size.")
print("In distributed training, each GPU does this with its local batch.")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_27_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Demonstrated the batch size problem** -- larger batches converge slower per epoch if the LR is not adjusted

2. **Tested learning rate scaling rules** -- linear scaling (aggressive) and sqrt scaling (conservative) both improve large-batch training

3. **Shown the importance of warmup** -- large learning rates cause instability without a warmup period

4. **Measured gradient noise** -- confirmed that gradient variance decreases as 1/batch_size, explaining why larger LRs are justified

5. **Simulated multi-GPU scaling** -- demonstrated that with proper scaling rules, we can scale to many GPUs effectively

### Key Takeaways

- When scaling batch size by k, scale LR by k (linear) or sqrt(k) (conservative)
- Always use warmup for large-batch training (1-5% of total steps)
- There is a critical batch size beyond which adding more GPUs gives diminishing returns
- Gradient accumulation lets you simulate large batches on small GPUs
- The sweet spot is usually at or below the critical batch size

### Next Notebook

In Notebook 3, we will use PyTorch's built-in profiler to identify bottlenecks in training loops -- compute, memory, communication, and data loading.